# 实现注意力机制
在上一章中，我们学会了把原始的文本数据转换成整LLM可以处理的向量表示。具体过程为：把原始文本拆分成token，为token创建对应的词表索引，并把token索引转换成向量表示。现在我们已经准备好了输入数据，接下来就可以实现LLM的核心组件：注意力机制了。
我们本章将会实现四种注意力机制变体，逐步深入的学习注意力机制，最终目标为实现一个简单高效的多头注意力机制，并在后续章节将其整合到LLM框架中


In [33]:
# 本笔记所用软件包
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.11.0


## 3.1 长序列建模问题
长序列建模问题是指在处理长文本序列时，模型难以捕捉到远距离依赖关系的问题。传统的RNN和LSTM等循环神经网络在处理长序列时会遇到梯度消失或爆炸的问题，导致模型难以学习到长距离的依赖关系。而Transformer模型通过引入自注意力机制，能够更好地捕捉长距离依赖关系，但在处理非常长的序列时仍然存在计算资源消耗过大的问题。
为了解决长序列建模问题，研究人员提出了多种注意力机制变体，如局部注意力、稀疏注意力、分层注意力和记忆增强注意力等。这些变体通过限制注意力的计算范围或引入额外的记忆机制，来降低计算复杂度并提高模型在长序列上的性能。

- 由于源语言和目标语言之间的语法结构存在差异，逐字翻译文本是不可行的：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="400px">

- 在引入 Transformer 模型之前，编码器-解码器 RNN 通常用于机器翻译任务- 在这种设置中，编码器处理来自源语言的标记序列，使用隐藏状态（神经网络中的一种中间层）来生成整个输入序列的压缩表示：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="500px">



## 3.2通过注意力机制捕捉长距离依赖关系
注意力机制是一种允许模型在处理输入序列时动态地关注不同部分的方法。它通过计算输入序列中每个位置与当前处理位置之间的相关性来实现这一点。注意力机制的核心思想是为输入序列中的每个位置分配一个权重，这些权重表示该位置对于当前处理位置的重要性。通过这种方式，模型能够捕捉到长距离依赖关系，因为它可以直接关注输入序列中的任何位置，而不受距离的限制。
注意力机制的计算通常涉及三个主要组件：查询（Query）、键（Key）和值（Value）。查询表示当前处理位置的表示，键和值分别表示输入序列中每个位置的表示。通过计算查询与键之间的相关性，模型可以得到一个权重分布，然后使用这些权重对值进行加权求和，得到当前处理位置的输出表示。这种机制使得模型能够灵活地关注输入序列中的不同部分，从而更好地捕捉长距离依赖关系。


## 3.3 通过自注意力关注输入的不同部分


### 3.3.1 一种无需可训练权重的简单自注意力机制

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="500px">

本小节实现一个简化的自注意力机制，其中查询（Q）、键（K）和值（V）都是输入序列的相同表示。我们将使用点积来计算查询和键之间的相关性，并使用softmax函数将相关性转换为权重分布。最后，我们将使用这些权重对值进行加权求和，得到当前处理位置的输出表示。

假设有一个简单的句子You journey starts with one step，在这个例子中每个token对应着一个三维的嵌入向量（转换过程为第二章的内容）。

在自注意力机制中，我们的目标是为输入序列的每个token计算一个新的上下文向量，这个新的向量将包含输入序列中与当前处理位置相关的信息。该权重应当是可学习的（本小节暂不实现），通过学习权重，模型可以动态地调整关注输入序列中不同部分的程度，从而更好地捕捉长距离依赖关系。

下面是简单的自注意力机制的实现：


In [34]:
import torch

#假设输入序列如下
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

#以第二个词为示例，计算它与输入序列中每个词的相关性
query = inputs[1]  # journey 的嵌入向量
# 创建一个空的张量，用于存储每个词与journey的相似度
#（这里用点积运算来简单代替）
attn_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # 计算journey与每个词的相似度

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


这里对注意力得分的计算使用了简单的点积运算，实际中我们通常会使用可学习的权重矩阵来计算Q查询、K键和V值，以便模型能够动态地调整关注输入序列中不同部分的程度。


对于计算出的注意力得分需要进行归一化处理，以便将它们转换为一个概率分布，这样我们就可以使用这些权重对值进行加权求和。通常我们会使用softmax函数来实现这个归一化过程：
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="500px">

In [35]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

# 使用softmax函数进行归一化
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())
# torch的softmax函数
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


得到权重之后就可以使用这些权重对输入序列中的值进行加权求和，得到当前处理位置的输出表示：

In [36]:
query = inputs[1] # journey 的嵌入向量

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### 3.3.2 计算所有位置的自注意力
上面的示例仅计算了输入序列中第二个词（journey）的自注意力。实际上，我们需要为输入序列中的每个位置计算自注意力，以便模型能够捕捉到输入序列中所有位置之间的相关性。为此，我们可以使用一个循环来计算每个位置的自注意力，或者更高效地使用矩阵运算来同时计算所有位置的自注意力。
下面是一个使用矩阵运算来同时计算所有位置的自注意力的示例：

In [37]:
# 创建一个空的矩阵，用于存储每个位置与输入序列中每个位置的相似度
attn_scores = torch.empty(6, 6)

# 计算每个位置与输入序列中每个位置的相似度
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [38]:
#矩阵乘法会更简单
attn_scores = inputs @ inputs.T

print(attn_scores)

#爱因斯坦求和约定（Einstein summation convention）也可以实现同样的功能
attn_scores_einsum = torch.einsum('id,jd->ij', inputs, inputs)
print(attn_scores_einsum)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [39]:
#然后使用softmax进行标准化
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)
#每行的和为1
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


和上一小节的方法一样，计算每个位置的上下文向量

In [40]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## 3.4 使用可训练权重计算查询、键和值
在前面的示例中，我们直接使用输入序列的嵌入向量作为查询、键和值来计算注意力得分和上下文向量。
然而，在实际的Transformer模型中，我们通常会使用可训练的权重矩阵来计算查询、键和值，以便模型能够动态地调整关注输入序列中不同部分的程度。

具体来说，我们会为查询、键和值分别定义三个权重矩阵W_Q、W_K和W_V，然后通过矩阵乘法将输入序列的嵌入向量转换为查询、键和值的表示。这样，模型就可以通过学习这些权重矩阵来动态地调整关注输入序列中不同部分的程度，从而更好地捕捉长距离依赖关系。
下面是一个使用可训练权重矩阵来计算查询、键和值的示例：

In [41]:
# x_2 = inputs[1]  # journey 的嵌入向量
d_in = inputs.shape[1]
d_out = 2 # 输出维度


然后我们使用torch的Parameter来定义可训练的权重矩阵W_Q、W_K和W_V：


In [42]:
torch.manual_seed(123)

W_Q = torch.nn.Parameter(torch.rand(d_in, d_out))
W_K = torch.nn.Parameter(torch.rand(d_in, d_out))
W_V = torch.nn.Parameter(torch.rand(d_in, d_out))

#计算对应位置的Q、K和V
Q = inputs @ W_Q
K = inputs @ W_K
V = inputs @ W_V

print("Query:", Q)
print("Key:", K)
print("Value:", V)

Query: tensor([[0.2309, 1.0966],
        [0.4306, 1.4551],
        [0.4300, 1.4343],
        [0.2355, 0.7990],
        [0.2983, 0.6565],
        [0.2568, 1.0533]], grad_fn=<MmBackward0>)
Key: tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)
Value: tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], grad_fn=<MmBackward0>)


拥有Q、K和V之后，我们就可以使用它们来计算注意力得分、权重和上下文向量了。具体来说，我们会使用查询Q与键K之间的点积来计算注意力得分，然后使用softmax函数将得分转换为权重，最后使用这些权重对值V进行加权求和，得到当前处理位置的上下文向量。下面是一个使用可训练权重矩阵来计算注意力得分、权重和上下文向量的示例：

In [43]:
attn_scores = Q @ K.T
print("Attention scores:", attn_scores)
d_k = K.shape[-1]
#这里归一化时使用d_k的平方根来缩放注意力分数，以避免在softmax函数中出现数值不稳定的问题。
attn_weights = torch.softmax(attn_scores/d_k**0.5, dim=-1)
print("Attention weights:", attn_weights)

Attention scores: tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]],
       grad_fn=<MmBackward0>)
Attention weights: tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]],
       grad_fn=<SoftmaxBackward0>)


In [44]:
#最后根据计算出的权重对值进行加权求和，得到上下文向量
context_vec = attn_weights @ V
print("Context vector:", context_vec)

Context vector: tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


### 3.4.2封装上述过程为一个类
我们可以将上述计算查询、键和值以及注意力得分、权重和上下文向量的过程封装到一个类中，以便在后续章节中更方便地使用这个注意力机制。下面是一个简单的封装示例：

In [45]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_Q = nn.Parameter(torch.rand(d_in, d_out))
        self.W_K   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_V = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_K
        queries = x @ self.W_Q
        values = x @ self.W_V

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="400px">

我们可以使用 PyTorch 的liner层简化上面的实现，如果我们禁用偏置单元，则相当于矩阵乘法- 与我们的手动“nn.Parameter(torch.rand(...)”方法相比，使用“nn.Linear”的另一个大优点是“nn.Linear”具有首选的权重初始化方案，这会导致更稳定的模型训练。


In [48]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


两个版本输出不同，这是因为我们在第一个版本中直接使用了随机初始化的权重矩阵W_Q、W_K和W_V，而在第二个版本中使用了nn.Linear层，这些层具有首选的权重初始化方案。这种初始化方案通常会导致更稳定的模型训练，因此在实际应用中，我们更倾向于使用nn.Linear层来实现注意力机制。

如果把v1的权重矩阵初始化和v2相同的矩阵，二者的输出将完全一致


## 3.5 使用因果注意力机制来屏蔽后续词
本小节把前面的标准自注意力机制修改为因果注意力机制（causal self-attention），以便在处理序列时屏蔽后续词。这样，模型在计算当前处理位置的上下文向量时，只能关注输入序列中当前处理位置之前的词，从而避免了信息泄露的问题。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="400px">

### 3.5.1 应用因果注意力掩码

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="600px">

前面的小节实现了自注意力机制。
最后的到的结果是每一个位置对于其他位置的注意力权重。



In [52]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [69]:
# 为了实现因果注意力，我们需要在计算注意力权重之前应用一个掩码（mask），这个掩码会将输入序列中当前处理位置之后的词的注意力得分设置为负无穷大（-inf），这样在softmax函数中这些位置的权重将会变为0，从而屏蔽后续词的影响。
seq_len = attn_scores.size(-1)

mask = torch.tril(torch.ones(seq_len, seq_len))

attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2 使用dropout遮掩额外的注意力权重
dropout在深度学习模型中，一般用于防止过拟合。集体操作为在训练过程中，将一些神经元从网络中排除，从而防止过拟合。
在transformer模型中，dropout通常应用于注意力权重，以增加模型的鲁棒性并防止过拟合。通过在训练过程中随机遮掩一些注意力权重，模型可以更好地泛化到未见过的数据，从而提高其性能。



In [67]:
#如果我们应用 0.5 (50%) 的 dropout 率，则未丢弃的值将相应地缩放 1/0.5 = 2- 缩放比例由公式 1 / (1 - `dropout_rate`) 计算
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [70]:
#把dropout应用到注意力权重上
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


### 3.5.3 实现一个简洁的因果注意力类

本小节我们把因果掩码和dropout整合到前面实现的自注意力类中，创建一个简洁的因果注意力类。这个类将包含计算查询、键和值的权重矩阵，以及应用因果掩码和dropout的功能。

回顾一下第二章，我们的数据是多个输入组成的批次，我们要确定代码可以处理多个输入。



In [71]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

torch.Size([2, 6, 3])


In [72]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        #与之前的SelfAttention_v1类相比，新增了dropout层。
        self.dropout = nn.Dropout(dropout)
        #buffer的作用是避免设备不匹配的错误（cpu，gpu）
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


dropout 仅在训练期间应用，而不在推理期间应用

## 3.6 从单头注意力拓展到多头注意力
多头的意思是我们有多个独立的注意力机制（头），每个头都有自己的一套权重矩阵W_Q、W_K和W_V。每个头会计算自己的注意力得分、权重和上下文向量，然后我们将所有头的上下文向量连接起来，并通过一个线性层进行投影，得到最终的输出表示。
多头注意力机制的优点在于它允许模型在不同的子空间中捕捉输入序列中的不同类型的相关性，从而提高模型的表达能力和性能。通过使用多个头，模型可以同时关注输入序列中的不同部分，从而更好地捕捉长距离依赖关系和复杂的模式。
### 3.6.1 堆叠多层单头注意力层

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="400px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="400px">
堆叠多层单头注意力层，可以理解为多个单头注意力层堆叠在一起，每个单头注意力层处理输入序列的局部信息，并生成一个表示该局部信息的向量。然后，将这些向量堆叠在一起，生成一个表示输入序列的全局信息向量。


In [73]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


- 在上面的实现中，嵌入维度为 4，如果我们将“d_out=2”作为键、查询、值向量以及上下文向量的嵌入维度。由于我们有 2 个注意力头，因此我们的输出嵌入维度为 2*2=4

### 3.6.2 分割权重矩阵以实现多头注意力

虽然上面是多头注意力的直观且功能齐全的实现（包装了早期的单头注意力“CausalAttention”实现），但我们可以编写一个名为“MultiHeadAttention”的独立类来实现相同的- 我们不会为这个独立的“MultiHeadAttention”类连接单个注意力头- 相反，我们创建单个W_query，W_key和W_value权重矩阵，然后将它们分成每个参与者的单独矩阵离子头：

In [74]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


两个实现本质上完全一样
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="400px">

## 3.7本章总结
本章我们实现了多头注意力机制，这是Transformer模型的核心组件之一。我们首先实现了一个简化的自注意力机制，然后逐步引入了因果掩码和dropout，最后实现了一个完整的多头注意力类。通过这个过程，我们深入理解了注意力机制的计算过程以及如何在实际中应用它来捕捉输入序列中的长距离依赖关系。在后续章节中，我们将继续构建LLM框架，并将多头注意力机制整合到其中，以实现更强大的语言模型。

（本章较为重要，可以多花时间多学习细节）